# 06 — SVM Gradients and Numerical Gradient Checking

The SVM loss tells us how poor a set of scores is. To improve the parameters, we also need the direction in which the loss changes. In this notebook we derive that direction, propagate it through a linear classifier, and check the result numerically.

## Learning objectives

- derive the multiclass SVM gradient with respect to class scores;
- propagate score gradients to `W` and `b` with the chain rule;
- include the L2 regularization gradient;
- implement centered-difference numerical gradients;
- interpret relative error and recognize hinge-loss nondifferentiability.


In [1]:
from pathlib import Path

import numpy as np
from torchvision.datasets import CIFAR10

SEED = 42
rng = np.random.default_rng(SEED)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

## 1. From active margins to score gradients

For example $i$ and an incorrect class $j$, the margin is

$$m_{ij} = \max(0, s_{ij} - s_{iy_i} + \Delta).$$

An **active margin** is one with $m_{ij} > 0$. Each active margin contributes:

- $+1$ to the derivative with respect to the incorrect score $s_{ij}$;
- $-1$ to the derivative with respect to the correct score $s_{iy_i}$.

Therefore, if a row has $k$ active incorrect margins, its correct-score derivative is $-k$. Inactive margins contribute zero. Finally, divide all entries by batch size $N$ because our data loss is a mean.

### Exercise 1 — Work through one row by hand

For scores `[3.2, 5.1, -1.7]`, correct class `0`, and $\Delta=1$:

1. calculate the two incorrect-class margins;
2. identify which margins are active;
3. write the unaveraged score-gradient row `[ds_0, ds_1, ds_2]`;
4. explain why its entries sum to zero.

Write your calculation in a new Markdown cell, then complete the values below.

In [6]:
scores_one = np.array([3.2, 5.1, -1.7])
label_one = 0
delta = 1.0

margins_one = np.maximum(0, scores_one - scores_one[label_one] + delta)
margins_one[label_one] = 0.0
score_gradient_one = np.zeros_like(scores_one)
for i in range(len(scores_one)):
    if i == label_one:
        continue
    if margins_one[i] > 0:
        score_gradient_one[i] = 1.0
        score_gradient_one[label_one] -= 1.0

np.testing.assert_allclose(margins_one, np.array([0.0, 2.9, 0.0]))
np.testing.assert_allclose(score_gradient_one, np.array([-1.0, 1.0, 0.0]))
assert np.isclose(score_gradient_one.sum(), 0.0)
print("Margins:", margins_one)
print("Score gradient:", score_gradient_one)

Margins: [0.  2.9 0. ]
Score gradient: [-1.  1.  0.]


## 2. Vectorized score gradient

Start with an indicator matrix whose incorrect-class entry is `1` exactly where its margin is positive. The correct-class entry in each row must then become the negative number of active incorrect margins in that row.

A useful invariant follows: every row of `d_scores` sums to zero. Adding the same constant to every class score leaves all score differences—and therefore the loss—unchanged.

In [8]:
scores_small = np.array([
    [3.2, 5.1, -1.7],
    [1.3, 2.0, 4.2],
    [0.5, 0.1, -0.2],
])
labels_small = np.array([0, 2, 1])


def svm_margins(scores: np.ndarray, labels: np.ndarray, delta: float = 1.0) -> np.ndarray:
    """Return nonnegative SVM margins, with correct-class entries set to zero."""
    correct_scores = scores[np.arange(scores.shape[0]), labels][:, None]
    margins = np.maximum(0.0, scores - correct_scores + delta)
    margins[np.arange(scores.shape[0]), labels] = 0.0
    return margins


margins_small = svm_margins(scores_small, labels_small)
margins_small

array([[0. , 2.9, 0. ],
       [0. , 0. , 0. ],
       [1.4, 0. , 0.7]])

### Exercise 2 — Implement `svm_score_gradient`

Use the already-computed margin matrix and labels. Do not use a Python loop. Remember that the returned gradient corresponds to the **mean** batch loss.

In [9]:
def svm_score_gradient(margins: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Return the gradient of mean SVM data loss with respect to scores."""
    # create the active-margin indicator matrix.
    active_margins = (margins > 0).astype(float)
    # put the negative active-margin count in each correct-class entry.
    active_margins[np.arange(margins.shape[0]), labels] = -active_margins.sum(axis=1)
    # average over examples.
    return active_margins / margins.shape[0]


d_scores_small = svm_score_gradient(margins_small, labels_small)
expected_d_scores = np.array([
    [-1.0, 1.0, 0.0],
    [0.0, 0.0, 0.0],
    [1.0, -2.0, 1.0],
]) / 3

np.testing.assert_allclose(d_scores_small, expected_d_scores)
np.testing.assert_allclose(d_scores_small.sum(axis=1), 0.0)
print(d_scores_small)

[[-0.33333333  0.33333333  0.        ]
 [ 0.          0.         -0.        ]
 [ 0.33333333 -0.66666667  0.33333333]]


## 3. Chain rule: from scores to parameters

For a batch, our linear classifier is

$$S = XW^T + b,$$

with shapes

- $X$: `(N, D)`;
- $W$: `(C, D)`;
- $b$: `(C,)`;
- $S$ and $dS$: `(N, C)`.

Following the paths through the matrix multiplication gives

$$dW = dS^T X, \qquad db = \sum_{i=1}^{N} dS_i.$$

For total loss $L = L_{data} + \lambda\sum W^2$, add $2\lambda W$ to `dW`. We do not regularize the bias.

### What are $S$, $dS$, $dW$, and $db$?

$S$ is the matrix of scores produced during the forward pass. Entry $S_{ij}$ is the score assigned to class $j$ for example $i$. The letter `d` means **derivative of the final scalar loss with respect to** a quantity:

$$dS_{ij} = \frac{\partial L}{\partial S_{ij}}, \qquad dW_{jd} = \frac{\partial L}{\partial W_{jd}}, \qquad db_j = \frac{\partial L}{\partial b_j}.$$

Thus, `dS` is not a second score matrix. It tells us how a tiny increase in each score would change the loss. Positive means increasing that score increases the loss; negative means increasing it decreases the loss. `dW` and `db` give the same information for the model parameters.

For one score, expand the linear-classifier formula into scalars:

$$S_{ij} = \sum_{d=1}^{D} X_{id}W_{jd} + b_j.$$

A particular weight $W_{jd}$ affects class-$j$ scores for every example. By the chain rule, collect all of those paths:

$$\frac{\partial L}{\partial W_{jd}} = \sum_i \frac{\partial L}{\partial S_{ij}}\frac{\partial S_{ij}}{\partial W_{jd}} = \sum_i dS_{ij}X_{id}.$$

This scalar equation for every $j,d$ is exactly the matrix multiplication `dW = dS.T @ X`. Since $\partial S_{ij}/\partial b_j=1$, the bias formula is `db = dS.sum(axis=0)`.

A small integer example makes the shapes concrete:

```text
X  = [[1, 2],       dS = [[-1,  1],
      [3, 4]]             [ 2, -2]]

dW = dS.T @ X = [[ 5,  6],
                  [-5, -6]]
db = rows of dS summed = [1, -1]
```

For example, `dW[0, 0] = (-1 × 1) + (2 × 3) = 5`: class `0`'s first weight receives a contribution from both examples.

### Where does the factor of 2 in regularization go?

The factor comes from differentiating a square: $d(w^2)/dw=2w$. Two conventions are common, and both are valid if the loss and gradient match:

| Regularization loss | Regularization gradient |
|---|---|
| $\lambda\sum W^2$ | $2\lambda W$ |
| $\frac{1}{2}\lambda\sum W^2$ | $\lambda W$ |

The one-half is sometimes placed in the **loss** purely to cancel the `2` in its derivative. It does not mean regularization is fundamentally different. In this project we use the first convention, matching the preceding SVM-loss notebook: `reg_loss = regularization_strength * np.sum(W * W)` and `dW += 2 * regularization_strength * W`. Changing only one of these two lines would make the analytic gradient incorrect, and numerical gradient checking would expose it.

### Exercise 3 — Implement loss and analytic gradients

Complete the vectorized function. Keep intermediate shapes visible while working; each formula should be predictable before it is run.

In [10]:
def svm_loss_and_gradient(
    X: np.ndarray,
    y: np.ndarray,
    W: np.ndarray,
    b: np.ndarray,
    regularization_strength: float = 0.0,
    delta: float = 1.0,
) -> tuple[float, np.ndarray, np.ndarray]:
    """Return total SVM loss and analytic gradients for W and b."""
    if X.ndim != 2 or W.ndim != 2:
        raise ValueError("X and W must be 2D arrays.")
    if y.ndim != 1 or b.ndim != 1:
        raise ValueError("y and b must be 1D arrays.")
    if X.shape[0] != y.shape[0]:
        raise ValueError("X and y must contain the same number of examples.")
    if X.shape[1] != W.shape[1] or W.shape[0] != b.shape[0]:
        raise ValueError("The feature and class dimensions do not match.")
    if X.shape[0] == 0:
        raise ValueError("The batch must not be empty.")
    if not np.issubdtype(y.dtype, np.integer):
        raise TypeError("Labels must contain integers.")
    if np.any((y < 0) | (y >= W.shape[0])):
        raise ValueError("Labels must be valid class indices.")
    if regularization_strength < 0 or delta <= 0:
        raise ValueError("Regularization must be nonnegative and delta positive.")

    # scores, margins, data loss, and regularization loss.
    scores = X @ W.T + b
    margins = svm_margins(scores, y, delta)
    data_loss = np.mean(np.sum(margins, axis=1))
    reg_loss = regularization_strength * np.sum(W * W)
    # d_scores, dW (including regularization), and db.
    d_scores = svm_score_gradient(margins, y)
    dW = d_scores.T @ X + 2 * regularization_strength * W
    db = d_scores.sum(axis=0)
    return data_loss + reg_loss, dW, db


X_tiny = np.array([[1.0, -2.0], [0.5, 1.5], [-1.0, 2.0]])
# Repeating a class here keeps the tiny bias gradients away from zero, which
# makes their relative-error comparison easier to interpret.
y_tiny = np.array([0, 0, 0])
W_tiny = rng.normal(0.0, 0.1, size=(3, 2))
b_tiny = rng.normal(0.0, 0.1, size=3)

loss_tiny, dW_tiny, db_tiny = svm_loss_and_gradient(
    X_tiny, y_tiny, W_tiny, b_tiny, regularization_strength=0.1
)
assert np.ndim(loss_tiny) == 0
assert dW_tiny.shape == W_tiny.shape
assert db_tiny.shape == b_tiny.shape
print("Loss:", loss_tiny)
print("dW:\n", dW_tiny)
print("db:", db_tiny)

Loss: 2.0009404803755086
dW:
 [[-0.33028616 -1.01039984]
 [ 0.17417118  0.50940565]
 [ 0.14715631  0.4869782 ]]
db: [-2.  1.  1.]


## 4. Numerical gradients

An analytic gradient comes from calculus and code. A numerical gradient estimates the slope by perturbing one parameter coordinate at a time:

$$\frac{\partial L}{\partial \theta_k} \approx \frac{L(\theta_k+h)-L(\theta_k-h)}{2h}.$$

This **centered difference** is generally more accurate than using only $L(\theta_k+h)-L(\theta_k)$. Numerical gradients are slow, so they are a debugging tool for small problems—not a training method.

### Exercise 4 — Implement a numerical gradient

`loss_function` takes no arguments and reads `parameter` through its closure. Perturb every coordinate, evaluate both sides, restore the original value, and store the centered difference.

In [11]:
def numerical_gradient(
    loss_function, parameter: np.ndarray, h: float = 1e-5
) -> np.ndarray:
    """Estimate the gradient of a scalar loss with centered differences."""
    if h <= 0:
        raise ValueError("h must be positive.")

    gradient = np.zeros_like(parameter, dtype=np.float64)
    # loop over np.ndindex(parameter.shape).
    for index in np.ndindex(parameter.shape):
        original_value = parameter[index]
        # evaluate loss at original + h and original - h.
        parameter[index] = original_value + h
        loss_plus_h = loss_function()
        parameter[index] = original_value - h
        loss_minus_h = loss_function()
        gradient[index] = (loss_plus_h - loss_minus_h) / (2 * h)
        # restore the coordinate even after both evaluations.
        parameter[index] = original_value  # Restore the original value.

    return gradient


quadratic_parameter = np.array([1.5, -2.0], dtype=np.float64)
quadratic_loss = lambda: np.sum(quadratic_parameter**2)
quadratic_numeric = numerical_gradient(quadratic_loss, quadratic_parameter)
np.testing.assert_allclose(quadratic_numeric, 2 * quadratic_parameter, rtol=1e-6, atol=1e-8)
np.testing.assert_allclose(quadratic_parameter, np.array([1.5, -2.0]))
print("Numerical:", quadratic_numeric)
print("Analytic: ", 2 * quadratic_parameter)

Numerical: [ 3. -4.]
Analytic:  [ 3. -4.]


## 5. Compare analytic and numerical gradients

Absolute error alone can be misleading because gradient magnitudes vary. We use

$$\text{relative error} = \frac{|g_a-g_n|}{\max(10^{-8}, |g_a|+|g_n|)},$$

where $g_a$ and $g_n$ are analytic and numerical gradients. For this tiny float64 problem, maximum relative errors should usually be below roughly $10^{-6}$.

In [12]:
def max_relative_error(analytic: np.ndarray, numerical: np.ndarray) -> float:
    denominator = np.maximum(1e-8, np.abs(analytic) + np.abs(numerical))
    return float(np.max(np.abs(analytic - numerical) / denominator))


regularization_strength = 0.1
loss_tiny, dW_tiny, db_tiny = svm_loss_and_gradient(
    X_tiny, y_tiny, W_tiny, b_tiny, regularization_strength
)
loss_from_current_parameters = lambda: svm_loss_and_gradient(
    X_tiny, y_tiny, W_tiny, b_tiny, regularization_strength
)[0]

dW_numerical = numerical_gradient(loss_from_current_parameters, W_tiny)
db_numerical = numerical_gradient(loss_from_current_parameters, b_tiny)

W_error = max_relative_error(dW_tiny, dW_numerical)
b_error = max_relative_error(db_tiny, db_numerical)
print(f"W maximum relative error: {W_error:.3e}")
print(f"b maximum relative error: {b_error:.3e}")
assert W_error < 1e-6
assert b_error < 1e-6

W maximum relative error: 6.865e-11
b maximum relative error: 7.827e-12


### Why can a correct implementation sometimes fail a gradient check?

The hinge $\max(0, m)$ has a sharp corner at $m=0$, so its derivative is not uniquely defined there. If a perturbation crosses that corner, the numerical slope and the branch selected by the analytic implementation may disagree. Random continuous parameters usually keep us away from exact corners.

The step $h$ also matters: too large measures curvature over a broad interval; too small amplifies floating-point rounding. Use float64, a small problem, several coordinates, and a reasonable value such as `1e-5`.

### Exercise 5 — Investigate the step size

Run the same check for the values of $h$ below. Record what happens at the extremes and explain why the smallest numerical error need not occur at the smallest $h$.

In [13]:
for h in [1e-2, 1e-5, 1e-8, 1e-12]:
    dW_numeric_h = numerical_gradient(loss_from_current_parameters, W_tiny, h=h)
    error_h = max_relative_error(dW_tiny, dW_numeric_h)
    print(f"h={h:.0e}: maximum relative error={error_h:.3e}")

h=1e-02: maximum relative error=3.000e-14
h=1e-05: maximum relative error=6.865e-11
h=1e-08: maximum relative error=2.859e-08
h=1e-12: maximum relative error=3.841e-04


### Why is a smaller $h$ not always better?

Centered difference subtracts two nearby floating-point numbers:

$$g_h = \frac{L(\theta+h)-L(\theta-h)}{2h}.$$

There are two competing errors:

1. **Approximation error:** with a large $h$, the interval is too wide and the secant slope may not equal the local tangent slope. For a smooth function, centered difference usually reduces this error roughly like $h^2$.
2. **Floating-point rounding error:** with a tiny $h$, the two losses become almost identical. Their leading digits cancel during subtraction, leaving very few reliable digits, and division by tiny $2h$ magnifies that noise. This error grows roughly like machine precision divided by $h$.

So decreasing $h$ first helps and eventually hurts; there is a middle region with the best balance. In your output, error grows from about `7e-11` at `h=1e-5` to `4e-4` at `h=1e-12`, which is the rounding-error regime.

 `h=1e-2` result happens to be even better because the hinge loss is piecewise linear. As long as neither perturbation crosses a hinge corner, its centered-difference slope can be essentially exact even over that larger interval. That is a property of this particular point and function, not a general reason to prefer large $h$.

## 6. Sparse gradient check on CIFAR-10

A full numerical gradient for `W` would require two loss evaluations for every one of its `10 × 3072` entries. Instead, check a handful of randomly selected coordinates. This is faster while still testing the real image-data path.

We use a tiny batch, scale pixels, and keep all arrays in float64 for the check.

In [14]:
dataset = CIFAR10(root=DATA_DIR, train=True, download=True)
batch_indices = rng.choice(len(dataset.data), size=8, replace=False)
X_cifar = dataset.data[batch_indices].reshape(8, -1).astype(np.float64) / 255.0
X_cifar -= X_cifar.mean(axis=0, keepdims=True)
y_cifar = np.asarray(dataset.targets, dtype=np.int64)[batch_indices]
W_cifar = rng.normal(0.0, 1e-3, size=(10, X_cifar.shape[1]))
b_cifar = np.zeros(10, dtype=np.float64)

cifar_loss, dW_cifar, db_cifar = svm_loss_and_gradient(
    X_cifar, y_cifar, W_cifar, b_cifar, regularization_strength=1e-3
)
print("CIFAR-10 tiny-batch loss:", cifar_loss)
print("Shapes:", dW_cifar.shape, db_cifar.shape)

CIFAR-10 tiny-batch loss: 9.062204348268336
Shapes: (10, 3072) (10,)


### Exercise 6 — Check selected weight coordinates

Complete the helper for a single coordinate, then check ten random entries of `W_cifar`. A coordinate close to zero may have an unstable relative error even when its absolute difference is tiny, so print both values and interpret them together.

In [17]:
# rng.choice(W_cifar.size, size=10, replace=False)
np.unravel_index(17031, W_cifar.shape)

(np.int64(5), np.int64(1671))

In [18]:
def numerical_gradient_at_coordinate(
    loss_function, parameter: np.ndarray, index: tuple[int, ...], h: float = 1e-5
) -> float:
    """Estimate one coordinate of a gradient with a centered difference."""
    # perturb, evaluate, restore, and return the centered difference.
    original_value = parameter[index]
    parameter[index] = original_value + h
    loss_plus_h = loss_function()
    parameter[index] = original_value - h
    loss_minus_h = loss_function()
    parameter[index] = original_value  # Restore the original value.
    return (loss_plus_h - loss_minus_h) / (2 * h)


cifar_loss_function = lambda: svm_loss_and_gradient(
    X_cifar, y_cifar, W_cifar, b_cifar, regularization_strength=1e-3
)[0]

flat_indices = rng.choice(W_cifar.size, size=10, replace=False)
for flat_index in flat_indices:
    index = np.unravel_index(flat_index, W_cifar.shape)
    numeric = numerical_gradient_at_coordinate(cifar_loss_function, W_cifar, index)
    analytic = dW_cifar[index]
    relative = abs(analytic - numeric) / max(1e-8, abs(analytic) + abs(numeric))
    print(
        f"W{index}: analytic={analytic:+.7e}, numerical={numeric:+.7e}, "
        f"relative error={relative:.3e}"
    )

W(np.int64(7), np.int64(3067)): analytic=-3.7438713e-01, numerical=-3.7438712e-01, relative error=1.078e-10
W(np.int64(7), np.int64(1981)): analytic=+2.9963287e-01, numerical=+2.9963287e-01, relative error=6.647e-11
W(np.int64(4), np.int64(946)): analytic=-1.1387709e-06, numerical=-1.1387336e-06, relative error=1.640e-05
W(np.int64(1), np.int64(38)): analytic=+3.0269649e-01, numerical=+3.0269649e-01, relative error=2.656e-11
W(np.int64(5), np.int64(1151)): analytic=-2.7206086e-01, numerical=-2.7206086e-01, relative error=1.749e-10
W(np.int64(2), np.int64(843)): analytic=-7.6103390e-07, numerical=-7.6108009e-07, relative error=3.034e-05
W(np.int64(0), np.int64(2694)): analytic=-1.1519537e-01, numerical=-1.1519537e-01, relative error=3.455e-10
W(np.int64(6), np.int64(3052)): analytic=-2.3529134e-01, numerical=-2.3529134e-01, relative error=2.012e-10
W(np.int64(9), np.int64(1475)): analytic=+9.9341851e-07, numerical=+9.9333874e-07, relative error=4.015e-05
W(np.int64(5), np.int64(2084)): 

## Reflection

Add a Markdown cell answering:

1. Why does every active incorrect margin add `+1` to one score derivative and `-1` to the correct score derivative?
2. Why does every row of `d_scores` sum to zero?
3. Explain the shapes in `dW = d_scores.T @ X` without relying only on memorization.
4. Why is `db` a sum over examples?
5. Why is the regularization gradient `2 * regularization_strength * W` under our convention?
6. What different roles do analytic and numerical gradients play?
7. Why should gradient checking use float64 and a small batch?
8. What can cause a gradient check to fail even when the implementation is reasonable?

Next we will study softmax probabilities and cross-entropy loss, then derive and check their gradients.

1. An active margin is $S_{ij} - S_{iy_i} + \Delta$. Its derivative with respect to the incorrect score $S_{ij}$ is $+1$, while its derivative with respect to the correct score $S_{iy_i}$ is $-1$. Increasing an incorrect score makes the loss larger; increasing the correct score makes it smaller.

2. Every active margin adds one `+1` and one `-1` within the same row of `d_scores`, so these contributions cancel. The row sum is therefore zero. This also agrees with the fact that shifting every score for one example by the same amount does not change any margin.

3. `X` has shape `(N, D)` and `d_scores` has shape `(N, C)`. Each weight gradient must combine contributions from all `N` examples while retaining one value for every class-feature pair. Therefore, `d_scores.T` has shape `(C, N)`, and `(C, N) @ (N, D)` produces `dW` with shape `(C, D)`, matching `W`. Elementwise, $dW_{jd} = \sum_i dS_{ij}X_{id}$.

4. The same bias $b_j$ is added to the class-$j$ score of every example, and $\partial S_{ij}/\partial b_j=1$. The chain rule therefore collects the contribution from every example: $db_j = \sum_i dS_{ij}$.

5. Our regularization loss is $\lambda\sum W^2$. Because the derivative of $w^2$ is $2w$, its gradient is $2\lambda W$.

6. A numerical gradient is a slow, approximate debugging check for the analytic gradient. The analytic gradient is exact apart from floating-point arithmetic and efficient enough to use during training.

7. Gradient checking uses float64 because it subtracts nearly equal loss values and needs enough precision to preserve their small difference. It uses a small batch and usually only selected parameter coordinates because each numerical-gradient coordinate requires two forward loss evaluations.

8. A check can disagree near the hinge corner where a margin is zero and the derivative is not uniquely defined. It can also be affected by a step size that is too large or too small, float32 rounding, near-zero gradients that make relative error unstable, or stochastic behavior that changes between loss evaluations.